In [1]:
import polars as pl
import seaborn as sns
import datetime as dt 


In [2]:
# 'categorzar'-> 1. uso quando ler uma coluna string é mais custosa que coluna numérica
#                2. no Pandas: df['coluna'],astype('Category')
#                3. no Polars: acast(pl.Categorica1)
#                4. Mais otimização: ideia-> criar um dicionário pra mapear de forma 'global', ou seja, qqlr tabela desse arq poderia usar o mapeamento.

In [3]:
pl.Categories()
pl.Config.set_fmt_float('full')
pl.Config.set_float_precision(2)


polars.config.Config

In [4]:
ARQUIVO= r'C:\analise de dados\arquivosBF\dados_bronze\df_bf.parquet'

In [7]:
try:
    hora_inicio= dt.datetime.now()
    #criar um plano de execução
    df_bf_plano = pl.scan_parquet(ARQUIVO)
    df_lazy = df_bf_plano.select(
        [pl.col('NOME MUNICÍPIO'),
         pl.col('VALOR PARCELA'),
         pl.col('UF'),
         pl.col('MÊS REFERÊNCIA'),
         pl.col('MÊS COMPETÊNCIA'),
         ]
    ).with_columns(
        (pl.col('MÊS REFERÊNCIA').cast(pl.Utf8).str.strptime(pl.Date, format='%Y%m')
        ),
    )
    hora_fim = dt.datetime.now()
    tempo = hora_fim - hora_inicio
    print(f'Tempo de execução {tempo}')
except Exception as e:
    print(f'Erro ao obter os dados do parquet: {e}')

Tempo de execução 0:00:00.000181


In [8]:
COLUNA ='VALOR PARCELA'

df_analise = df_lazy.select(
    pl.col(COLUNA).mean().alias('Média'),
    pl.col(COLUNA).median().alias('Mediana'),
    pl.col(COLUNA).std().alias('Desvio Padrão'),
    pl.col(COLUNA).skew().alias('Assimetria'),
    pl.col(COLUNA).kurtosis().alias('Curtose'),
    pl.col(COLUNA).min().alias('Mínimo'),
    pl.col(COLUNA).quantile(0.25).alias('Q1'),
    pl.col(COLUNA).quantile(0.75).alias('Q3'),
    pl.col(COLUNA).max().alias('Máximo')
).collect()

df_analise

Média,Mediana,Desvio Padrão,Assimetria,Curtose,Mínimo,Q1,Q3,Máximo
f64,f64,f64,f64,f64,f64,f64,f64,f64
668.26,650.00,190.28,0.89,4.67,25.00,600.00,750.00,3938.00


In [21]:
q1 = df_analise['Q1']
q3 = df_analise['Q3']
IQR = q3 - q1

limite_inferior = q1 - 1.5* IQR
limite_superior = q3 + 1.5* IQR

df_valores = df_lazy.select(
    pl.col('VALOR PARCELA')
).collect()
import numpy as np 


outliers_inf = (df_valores < limite_inferior).sum()
outliers_sup = (df_valores > limite_superior).sum()
total_outliers = outliers_inf + outliers_sup

porcentagem_outliers_totais = (
    total_outliers/df_valores.shape[0])*100

porcentagem_outliers_superiores = (
    outliers_sup/df_valores.shape[0])*100

porcentagem_outliers_inferiores = (
    outliers_inf/df_valores.shape[0])*100

print(f'Porcentagem de outliers inferiores: {porcentagem_outliers_inferiores}')
print(f'Porcentagem de outliers superiores: {porcentagem_outliers_superiores}')


Porcentagem de outliers inferiores: shape: (1, 1)
┌───────────────┐
│ VALOR PARCELA │
│ ---           │
│ f64           │
╞═══════════════╡
│ 8.46          │
└───────────────┘
Porcentagem de outliers superiores: shape: (1, 1)
┌───────────────┐
│ VALOR PARCELA │
│ ---           │
│ f64           │
╞═══════════════╡
│ 4.42          │
└───────────────┘
